# MiniFASNet face PAD evaluation (Colab / Kaggle)

This notebook tests the same two-scale MiniFASNet ONNX model family used by the app. It evaluates face crops from your own camera captures against printed-photo and screen-replay attempts. It is an experiment, not a certified security test.

**Dataset format:** create a CSV with columns `path,label,x1,y1,x2,y2`; `label` must be `live`, `print`, or `screen`, and the box uses pixel coordinates in the original BGR image. Keep the whole camera frame around each face so the model sees the same context as the application. You may instead use face-cropped images and set the box to the full image dimensions.

For Google Colab, upload the CSV and image files or mount Google Drive. For Kaggle, set `DATA_CSV` to a path under `/kaggle/input/...`. Do not mix the same recording session between calibration and held-out evaluation. The notebook processes images in the selected Colab/Kaggle runtime; follow that provider's data-handling policies.

The source project describes MiniFASNet classes as class 1 = real; classes 0 and 2 = spoof. A face passes only if class 1 is the largest score and the live probability meets the selected threshold.

## Runtime setup

Select a GPU accelerator in Colab/Kaggle if you want to measure GPU inference. CPU inference is supported. If your runtime has no ONNX Runtime installed, run `%pip install -q onnxruntime` for CPU, or `%pip install -q onnxruntime-gpu` for a CUDA-capable runtime **before importing `onnxruntime`**. Restart the notebook kernel after changing runtime packages. The cell below reports actual ONNX providers; the notebook does not claim GPU use unless `CUDAExecutionProvider` is available.

In [ ]:
from pathlib import Path
import hashlib
import time
from urllib.request import urlopen

import cv2
import numpy as np
import pandas as pd
import onnxruntime as ort
import pandas as pd

print('OpenCV:', cv2.__version__)
print('ONNX Runtime:', ort.__version__)
print('Available providers:', ort.get_available_providers())

In [ ]:
MODEL_COMMIT = '584d4421d7ac42c59e640796f46e886b0095367a'
MODEL_FILES = {
    '2.7_80x80_MiniFASNetV2.onnx': '0CBE5CAEC95C31DE9D2EF845CB85407D76AECD1B6A2C0E343F7D35306BFBCCB8',
    '4_0_0_80x80_MiniFASNetV1SE.onnx': 'A25886A85CDCFA2C4EA23EDB71DE35F250C17827B4CADD253A972B28C80FDF1E',
}
MODEL_DIR = Path('pad_models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for filename, expected_hash in MODEL_FILES.items():
    target = MODEL_DIR / filename
    if not target.exists() or hashlib.sha256(target.read_bytes()).hexdigest().upper() != expected_hash:
        url = f'https://raw.githubusercontent.com/QingHeYang/Silent-Face-Anti-Spoofing-onnx/{MODEL_COMMIT}/onnx/{filename}'
        with urlopen(url, timeout=60) as response:
            artifact = response.read()
        actual_hash = hashlib.sha256(artifact).hexdigest().upper()
        if actual_hash != expected_hash:
            raise RuntimeError(f'SHA-256 check failed for {filename}: {actual_hash}')
        target.write_bytes(artifact)
    print('Verified:', target.name, hashlib.sha256(target.read_bytes()).hexdigest().upper())

In [ ]:
MODEL_SCALES = {
    '2.7_80x80_MiniFASNetV2.onnx': 2.7,
    '4_0_0_80x80_MiniFASNetV1SE.onnx': 4.0,
}
providers = (['CUDAExecutionProvider', 'CPUExecutionProvider']
             if 'CUDAExecutionProvider' in ort.get_available_providers()
             else ['CPUExecutionProvider'])
sessions = []
for filename, scale in MODEL_SCALES.items():
    session = ort.InferenceSession(str(MODEL_DIR / filename), providers=providers)
    sessions.append((scale, session, session.get_inputs()[0].name, session.get_outputs()[0].name))
    print(filename, 'active providers:', session.get_providers())

def expanded_crop(frame_bgr, bbox_xyxy, scale):
    h, w = frame_bgr.shape[:2]
    x1, y1, x2, y2 = map(float, bbox_xyxy)
    x1, x2 = np.clip([x1, x2], 0, w - 1)
    y1, y2 = np.clip([y1, y2], 0, h - 1)
    bw, bh = x2 - x1, y2 - y1
    if bw <= 0 or bh <= 0:
        raise ValueError('Bounding box must have positive width and height')
    scale = min(scale, (w - 1) / bw, (h - 1) / bh)
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    half_w, half_h = bw * scale / 2, bh * scale / 2
    left, top = max(0, int(np.floor(cx - half_w))), max(0, int(np.floor(cy - half_h)))
    right = min(w, int(np.ceil(cx + half_w)) + 1)
    bottom = min(h, int(np.ceil(cy + half_h)) + 1)
    crop = frame_bgr[top:bottom, left:right]
    if crop.size == 0:
        raise ValueError('Bounding box produced an empty crop')
    resized = cv2.resize(crop, (80, 80), interpolation=cv2.INTER_LINEAR)
    return np.expand_dims(np.transpose(resized.astype(np.float32), (2, 0, 1)), 0)

def predict_pad(frame_bgr, bbox_xyxy):
    fused = np.zeros(3, dtype=np.float64)
    started = time.perf_counter()
    for scale, session, input_name, output_name in sessions:
        logits = session.run([output_name], {input_name: expanded_crop(frame_bgr, bbox_xyxy, scale)})[0][0]
        logits = np.asarray(logits, dtype=np.float64)
        exp_logits = np.exp(logits - logits.max())
        fused += exp_logits / exp_logits.sum()
    fused /= len(sessions)
    return {'scores': fused, 'live_score': float(fused[1]),
            'class': ('paper', 'real', 'screen')[int(fused.argmax())],
            'latency_ms': (time.perf_counter() - started) * 1000}

## Point to your labeled camera data

The CSV needs `path,label,x1,y1,x2,y2`. Paths may be absolute or relative to `DATA_ROOT`. Labels are `live`, `print`, and/or `screen`. Use camera images for both bona-fide and attack samples; a passport photo scan by itself is not a representative camera presentation attack. Capture the print/screen while held at the expected attendance distance and under varied lighting.

In [ ]:
# Examples:
# Google Colab: upload/mount your files and set these paths to /content/...
# Kaggle: DATA_ROOT = Path('/kaggle/input/your-private-dataset')
DATA_ROOT = Path('/content/pad_dataset') if Path('/content').exists() else Path('.')
DATA_CSV = DATA_ROOT / 'labels.csv'
LIVE_THRESHOLD = 0.50  # starting point only; evaluate alternatives on a calibration split

if not DATA_CSV.is_file():
    raise FileNotFoundError(f'Create the labeled dataset CSV first: {DATA_CSV}')
data = pd.read_csv(DATA_CSV)
required = {'path', 'label', 'x1', 'y1', 'x2', 'y2'}
if not required.issubset(data.columns):
    raise ValueError(f'CSV must contain columns {sorted(required)}')
data['label'] = data['label'].astype(str).str.lower().str.strip()
if not set(data['label']).issubset({'live', 'print', 'screen'}):
    raise ValueError('Labels must be live, print, or screen')
print('Examples:', len(data), '| labels:', data['label'].value_counts().to_dict())
data.head()

In [ ]:
rows = []
for item in data.itertuples(index=False):
    image_path = Path(item.path)
    if not image_path.is_absolute():
        image_path = DATA_ROOT / image_path
    frame = cv2.imread(str(image_path))
    if frame is None:
        raise FileNotFoundError(f'Could not read image: {image_path}')
    bbox = (item.x1, item.y1, item.x2, item.y2)
    prediction = predict_pad(frame, bbox)
    is_live = prediction['class'] == 'real' and prediction['live_score'] >= LIVE_THRESHOLD
    rows.append({
        'path': str(image_path), 'truth': item.label,
        'predicted': 'live' if is_live else 'spoof',
        'model_class': prediction['class'], 'live_score': prediction['live_score'],
        'paper_score': prediction['scores'][0], 'screen_score': prediction['scores'][2],
        'latency_ms': prediction['latency_ms'],
    })
results = pd.DataFrame(rows)
results.head(20)

In [ ]:
is_bona_fide = results['truth'].eq('live')
is_predicted_live = results['predicted'].eq('live')
live_count = int(is_bona_fide.sum())
spoof_count = int((~is_bona_fide).sum())
false_rejects = int((is_bona_fide & ~is_predicted_live).sum())
false_accepts = int((~is_bona_fide & is_predicted_live).sum())
frr = false_rejects / live_count if live_count else float('nan')
far = false_accepts / spoof_count if spoof_count else float('nan')
balanced_accuracy = ((1 - frr) + (1 - far)) / 2 if live_count and spoof_count else float('nan')
print(f'Live samples: {live_count}; spoof samples: {spoof_count}')
print(f'False accepts: {false_accepts}; FAR: {far:.3f}')
print(f'False rejects: {false_rejects}; FRR: {frr:.3f}')
print(f'Balanced accuracy: {balanced_accuracy:.3f}')
print(f'Mean inference latency: {results.latency_ms.mean():.2f} ms/image')
print('\nPer-label outcomes:')
display(pd.crosstab(results['truth'], results['predicted'], margins=True))
print('Review errors:')
display(results[results['truth'].eq('live') != results['predicted'].eq('live')])

## Interpreting results

A low false-accept rate on a few examples does not establish security. Keep people and capture sessions separate between calibration and held-out testing; include different phones, monitors, print materials, brightness, viewing angles, and distances. Choose the decision threshold based on an acceptable false-accept/false-reject tradeoff, then rerun the held-out set without retuning. This passive RGB model may fail on unseen attacks and may reject genuine people. Combine it with a prompted challenge-response if the risk warrants it.

Model sources: [Minivision Silent-Face-Anti-Spoofing](https://github.com/minivision-ai/Silent-Face-Anti-Spoofing) and the Apache-2.0 [pinned ONNX conversion](https://github.com/QingHeYang/Silent-Face-Anti-Spoofing-onnx/tree/584d4421d7ac42c59e640796f46e886b0095367a).